In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [13]:
!pip install pandas numpy scikit-learn xgboost joblib

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb

# =========================
# 1. Load Data
# =========================
df = pd.read_csv('IPL.csv')

# =========================
# 2. Convert Ball Data → Match Data
# =========================
matches = []

for match_id, group in df.groupby('match_id'):
    first = group.iloc[0]
    
    teams = group['batting_team'].unique()
    if len(teams) != 2:
        continue
    
    t1, t2 = teams[0], teams[1]
    
    winner = first['match_won_by']
    
    # Skip invalid matches
    if pd.isna(winner):
        continue
    
    matches.append({
        'team1': t1,
        'team2': t2,
        'toss_winner': first['toss_winner'],
        'toss_decision': first['toss_decision'],
        'venue': first['venue'],
        'season': first['season'],
        'winner': 1 if winner == t1 else 0
    })

match_df = pd.DataFrame(matches)

# =========================
# 3. Data Cleaning
# =========================
# Convert season safely
match_df['season'] = pd.to_numeric(match_df['season'], errors='coerce')

# Drop bad rows
match_df = match_df.dropna()

# =========================
# 4. Encoding
# =========================
le_team = LabelEncoder()
le_venue = LabelEncoder()
le_toss = LabelEncoder()

# Fit on all teams together (IMPORTANT)
all_teams = pd.concat([
    match_df['team1'],
    match_df['team2'],
    match_df['toss_winner']
])

le_team.fit(all_teams)

match_df['team1_enc'] = le_team.transform(match_df['team1'])
match_df['team2_enc'] = le_team.transform(match_df['team2'])
match_df['toss_winner_enc'] = le_team.transform(match_df['toss_winner'])

match_df['venue_enc'] = le_venue.fit_transform(match_df['venue'])
match_df['toss_decision_enc'] = le_toss.fit_transform(match_df['toss_decision'])

# =========================
# 5. Features & Target
# =========================
X = match_df[[
    'team1_enc',
    'team2_enc',
    'toss_winner_enc',
    'toss_decision_enc',
    'venue_enc',
    'season'
]].astype(float)   # 🔥 ensure numeric

y = match_df['winner'].astype(int)

# =========================
# 6. Train Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 7. Train Model (XGBoost)
# =========================
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# =========================
# 8. Evaluation
# =========================
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

# =========================
# 9. Save Model
# =========================
import joblib
joblib.dump(model, "ipl_model.pkl")

Accuracy: 56.06 %


['ipl_model.pkl']

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb
import joblib

# =========================
# 1. Load Data
# =========================
df = pd.read_csv('IPL.csv')

# =========================
# 2. Convert to Match-Level Data
# =========================
matches = []

for match_id, group in df.groupby('match_id'):
    first = group.iloc[0]
    
    teams = group['batting_team'].unique()
    if len(teams) != 2:
        continue
    
    t1, t2 = teams[0], teams[1]
    winner = first['match_won_by']
    
    # ✅ FIX: Skip invalid winners
    if pd.isna(winner) or winner not in [t1, t2]:
        continue
    
    matches.append({
        'team1': t1,
        'team2': t2,
        'toss_winner': first['toss_winner'],
        'toss_decision': first['toss_decision'],
        'venue': first['venue'],
        'season': first['season'],
        'winner_team': winner
    })

match_df = pd.DataFrame(matches)

# =========================
# 3. Data Cleaning
# =========================
match_df['season'] = pd.to_numeric(match_df['season'], errors='coerce')
match_df = match_df.dropna()

# =========================
# 4. Feature Engineering 🔥
# =========================

# 🔥 Team Win Rate
team_stats = {}

for _, row in match_df.iterrows():
    t1, t2, winner = row['team1'], row['team2'], row['winner_team']
    
    for team in [t1, t2]:
        if team not in team_stats:
            team_stats[team] = {'wins': 0, 'matches': 0}
    
    team_stats[t1]['matches'] += 1
    team_stats[t2]['matches'] += 1
    
    # safety (already filtered but double safe)
    if winner in team_stats:
        team_stats[winner]['wins'] += 1

def get_win_rate(team):
    stats = team_stats.get(team, {'wins': 0, 'matches': 1})
    return stats['wins'] / stats['matches']

match_df['team1_win_rate'] = match_df['team1'].apply(get_win_rate)
match_df['team2_win_rate'] = match_df['team2'].apply(get_win_rate)

# 🔥 Head-to-head
h2h = {}

h2h_feature = []

for _, row in match_df.iterrows():
    t1, t2 = row['team1'], row['team2']
    winner = row['winner_team']
    
    key = tuple(sorted([t1, t2]))
    
    if key not in h2h:
        h2h[key] = {'t1_wins': 0, 't2_wins': 0}
    
    record = h2h[key]
    
    total = record['t1_wins'] + record['t2_wins']
    ratio = record['t1_wins'] / total if total != 0 else 0.5
    h2h_feature.append(ratio)
    
    if winner == t1:
        record['t1_wins'] += 1
    else:
        record['t2_wins'] += 1

match_df['h2h_ratio'] = h2h_feature

# =========================
# 5. Encoding
# =========================
le_team = LabelEncoder()
le_venue = LabelEncoder()
le_toss = LabelEncoder()

all_teams = pd.concat([
    match_df['team1'],
    match_df['team2'],
    match_df['toss_winner']
])

le_team.fit(all_teams)

match_df['team1_enc'] = le_team.transform(match_df['team1'])
match_df['team2_enc'] = le_team.transform(match_df['team2'])
match_df['toss_winner_enc'] = le_team.transform(match_df['toss_winner'])

match_df['venue_enc'] = le_venue.fit_transform(match_df['venue'])
match_df['toss_decision_enc'] = le_toss.fit_transform(match_df['toss_decision'])

# Target
match_df['winner'] = (match_df['winner_team'] == match_df['team1']).astype(int)

# =========================
# 6. Features & Target
# =========================
X = match_df[[
    'team1_enc',
    'team2_enc',
    'toss_winner_enc',
    'toss_decision_enc',
    'venue_enc',
    'season',
    'team1_win_rate',
    'team2_win_rate',
    'h2h_ratio'
]].astype(float)

y = match_df['winner']

# =========================
# 7. Train Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 8. Train Model
# =========================
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# =========================
# 9. Evaluation
# =========================
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

# =========================
# 10. Save Model
# =========================
joblib.dump(model, "ipl_model_advanced.pkl")

Accuracy: 49.74 %


['ipl_model_advanced.pkl']